In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, io, time, datetime
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
from tqdm.notebook import tqdm
from tabulate import tabulate
from IPython.display import display
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import Counter

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.2f}'.format)
print(f'Ready | {datetime.date.today()}')         

Ready | 2026-03-10


In [2]:
CONFIG = {
    'min_price'              : 10,
    'max_price'              : 500,
    'min_mcap_cr'            : 200,      # Min 200 Cr (small cap lower bound)
    'max_mcap_cr'            : 25000,    # Max 25000 Cr (mid cap upper bound)
    'tight_range_pct'        : 6.0,      # 10-day H-L range % threshold
    'adx_max'                : 15,       # ADX must be below this
    'rsi_min'                : 55,
    'rsi_max'                : 70,
    'vol_spike_mult'         : 3.0,      # Volume must be > 3x 20-day avg
    'delivery_pct_min'       : 40,       # Minimum delivery %
    'candle_body_pct'        : 60,       # Bullish candle body % threshold
    # Scoring weights (sum = 100)
    'w_vol_spike'            : 20,
    'w_rsi'                  : 15,
    'w_tight_range'          : 15,
    'w_adx'                  : 10,
    'w_atr_contract'         : 10,
    'w_price_vs_10dhigh'     : 10,
    'w_ema_align'            : 5,
    'w_candle_body'          : 5,
    'w_bb_squeeze'           : 5,
    'w_delivery'             : 5,
    # Output
    'top_n'                  : 15,
    'batch_size'             : 50,
    'sleep_between_batches'  : 1,
}

PORTFOLIO_VALUE        = 500_000   # Change to your capital in INR
MAX_RISK_PER_TRADE_PCT = 2.0       # % of portfolio risked per BTST trade

print('Configuration loaded.')

Configuration loaded.


In [3]:
NSE_EQUITY_URL = 'https://archives.nseindia.com/content/equities/EQUITY_L.csv'

HEADERS = {
    'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer'        : 'https://www.nseindia.com/',
}

session = requests.Session()
session.headers.update(HEADERS)

try:
    session.get('https://www.nseindia.com', timeout=10)
    time.sleep(1)
except Exception as e:
    print(f'Cookie prime failed: {e}')

def fetch_nse_equity_list():
    try:
        r = session.get(NSE_EQUITY_URL, timeout=20)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        df.columns = [c.strip() for c in df.columns]
        df = df[df['SERIES'].str.strip() == 'EQ'].copy()
        df['SYMBOL'] = df['SYMBOL'].str.strip()
        print(f'NSE equity list: {len(df)} EQ symbols')
        return df
    except Exception as e:
        print(f'Failed: {e}')
        return pd.DataFrame()

equity_df = fetch_nse_equity_list()
equity_df.head()

NSE equity list: 2117 EQ symbols


,SYMBOL,NAME OF COMPANY,SERIES,DATE OF LISTING,PAID UP VALUE,MARKET LOT,ISIN NUMBER,FACE VALUE
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,5,1,INE144J01027,5
1,21STCENMGM,21st Century Management Services Limited,EQ,03-MAY-1995,10,1,INE253B01015,10
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1,1,INE466L01038,1
3,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,10,1,INE748C01038,10
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,10,1,INE470A01017,10


In [4]:
def get_last_trading_day(offset=0):
    d = datetime.date.today() - datetime.timedelta(days=offset)
    while d.weekday() >= 5:
        d -= datetime.timedelta(days=1)
    return d

def fetch_bhavcopy(date):
    ds = date.strftime('%d%m%Y')
    url = f'https://archives.nseindia.com/products/content/sec_bhavdata_full_{ds}.csv'
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        df.columns = [c.strip() for c in df.columns]
        df['SYMBOL'] = df['SYMBOL'].str.strip()
        df = df[df['SERIES'].str.strip() == 'EQ'].copy()
        print(f'Bhavcopy {date}: {len(df)} records')
        return df
    except Exception as e:
        print(f'Bhavcopy failed {date}: {e}')
        return pd.DataFrame()

def fetch_delivery_data(date):
    ds = date.strftime('%d%m%Y')
    url = f'https://archives.nseindia.com/archives/equities/mto/MTO_{ds}.DAT'
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        data = []
        for line in r.text.strip().split('\n'):
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 7 and parts[0] == '20':
                data.append({'SYMBOL': parts[2], 'SERIES': parts[3],
                             'DELIV_PCT': float(parts[6]) if parts[6] else 0})
        df = pd.DataFrame(data)
        df = df[df['SERIES'] == 'EQ'][['SYMBOL', 'DELIV_PCT']].copy()
        print(f'Delivery {date}: {len(df)} records')
        return df
    except Exception as e:
        print(f'Delivery failed {date}: {e}')
        return pd.DataFrame(columns=['SYMBOL', 'DELIV_PCT'])

bhav_df = pd.DataFrame()
SCAN_DATE = None

for offset in range(0, 7):
    trade_date = get_last_trading_day(offset)
    bhav_df = fetch_bhavcopy(trade_date)
    if not bhav_df.empty:
        deliv_df  = fetch_delivery_data(trade_date)
        SCAN_DATE = trade_date
        break

print(f'Scan date: {SCAN_DATE}')

Bhavcopy failed 2026-03-10: 404 Client Error: Not Found for url: https://archives.nseindia.com/products/content/sec_bhavdata_full_10032026.csv
Bhavcopy 2026-03-09: 2432 records
Delivery 2026-03-09: 2432 records
Scan date: 2026-03-09


In [5]:
MIN_TRADED_VALUE_LAKH = 50

def build_universe(equity_df, bhav_df, deliv_df):
    if bhav_df.empty:
        return pd.DataFrame()
    col_map = {}
    for c in bhav_df.columns:
        cu = c.upper()
        if 'CLOSE' in cu and 'PREV' not in cu: col_map[c] = 'CLOSE'
        elif 'OPEN' in cu:                      col_map[c] = 'OPEN'
        elif 'HIGH' in cu:                      col_map[c] = 'HIGH'
        elif 'LOW'  in cu:                      col_map[c] = 'LOW'
        elif 'QTY'  in cu:                      col_map[c] = 'VOLUME'
        elif 'TURNOVER' in cu or 'VALUE' in cu: col_map[c] = 'TURNOVER_LACS'
    bhav_df = bhav_df.rename(columns=col_map)
    for col in ['CLOSE','OPEN','HIGH','LOW','VOLUME','TURNOVER_LACS']:
        if col in bhav_df.columns:
            bhav_df[col] = pd.to_numeric(bhav_df[col], errors='coerce')

    df = equity_df[['SYMBOL']].merge(bhav_df, on='SYMBOL', how='inner')
    df = df.merge(deliv_df, on='SYMBOL', how='left') if not deliv_df.empty else df.assign(DELIV_PCT=np.nan)

    if 'CLOSE' in df.columns:
        df = df[(df['CLOSE'] >= CONFIG['min_price']) & (df['CLOSE'] <= CONFIG['max_price'])]
    if 'TURNOVER_LACS' in df.columns:
        df = df[df['TURNOVER_LACS'] >= MIN_TRADED_VALUE_LAKH]

    print(f'Universe: {len(df)} symbols after pre-filter')
    return df.reset_index(drop=True)

universe_df = build_universe(equity_df, bhav_df, deliv_df)
universe_df[['SYMBOL','CLOSE','VOLUME','DELIV_PCT']].head(10)

Universe: 940 symbols after pre-filter


,SYMBOL,CLOSE,VOLUME,DELIV_PCT
0,20MICRONS,163.78,34372,51.83
1,3IINFOLTD,12.99,353355,81.49
2,5PAISA,296.80,17922,56.57
3,AADHARHFC,481.70,288703,17.65
4,AAKASH,11.28,1639693,21.03
5,AAREYDRUGS,69.41,44991,60.67
6,AARTIDRUGS,348.95,72807,51.85
7,AARTIIND,409.25,331791,46.12
8,AARTISURF,350.50,20669,66.69
9,ABCAPITAL,324.30,1739669,35.42


In [6]:
def download_ohlcv_batch(symbols, period='3mo'):
    yf_syms = [s + '.NS' for s in symbols]
    try:
        raw = yf.download(yf_syms, period=period, interval='1d',
                          group_by='ticker', auto_adjust=True,
                          progress=False, threads=True)
        result = {}
        for sym, yfsym in zip(symbols, yf_syms):
            try:
                df = raw.copy() if len(yf_syms) == 1 else raw[yfsym].copy()
                df.dropna(subset=['Close','Volume'], inplace=True)
                if len(df) >= 20:
                    result[sym] = df
            except Exception:
                pass
        return result
    except Exception as e:
        return {}

all_symbols  = universe_df['SYMBOL'].tolist()
batches      = [all_symbols[i:i+CONFIG['batch_size']] for i in range(0, len(all_symbols), CONFIG['batch_size'])]
ohlcv_cache  = {}

print(f'Downloading {len(all_symbols)} symbols in {len(batches)} batches...')
for batch in tqdm(batches, desc='OHLCV Download'):
    ohlcv_cache.update(download_ohlcv_batch(batch))
    time.sleep(CONFIG['sleep_between_batches'])

print(f'Downloaded: {len(ohlcv_cache)} symbols')

OHLCV Download:   0%|          | 0/19 [00:00<?, ?it/s]

Downloaded: 934 symbols


In [7]:
def fetch_mcap(symbols):
    cache = {}
    for sym in tqdm(symbols, desc='Market Cap'):
        try:
            mc = yf.Ticker(sym + '.NS').fast_info.market_cap
            cache[sym] = round(mc / 1e7, 2) if mc else None  # to Crores
        except Exception:
            cache[sym] = None
    return cache

mcap_cache = fetch_mcap(list(ohlcv_cache.keys()))

valid_symbols = [
    sym for sym in ohlcv_cache
    if mcap_cache.get(sym) is not None
    and CONFIG['min_mcap_cr'] <= mcap_cache[sym] <= CONFIG['max_mcap_cr']
]
print(f'After market-cap filter: {len(valid_symbols)} symbols')

Market Cap:   0%|          | 0/934 [00:00<?, ?it/s]

After market-cap filter: 824 symbols


In [8]:
def compute_indicators(df):
    if len(df) < 30:
        return None
    df = df.copy()
    df.columns = [c.lower() for c in df.columns]
    try:
        # ATR Contracting
        atr = ta.atr(df['high'], df['low'], df['close'], length=14)
        if atr is None or atr.dropna().empty: return None
        atr_today       = atr.iloc[-1]
        atr_contracting = atr_today < atr.iloc[-6] if len(atr) > 6 else False

        # ADX Turning Up
        adx_df  = ta.adx(df['high'], df['low'], df['close'], length=14)
        if adx_df is None or adx_df.empty: return None
        adx_col = [c for c in adx_df.columns if 'ADX' in c][0]
        adx_today      = adx_df[adx_col].iloc[-1]
        adx_turning_up = adx_today > adx_df[adx_col].iloc[-3]

        # RSI
        rsi       = ta.rsi(df['close'], length=14)
        rsi_today = rsi.iloc[-1] if rsi is not None else np.nan

        # EMA 20
        ema20             = ta.ema(df['close'], length=20)
        price_above_ema20 = df['close'].iloc[-1] > ema20.iloc[-1] if ema20 is not None else False

        # Bollinger Squeeze
        bb         = ta.bbands(df['close'], length=20, std=2.0)
        bb_squeeze = False
        if bb is not None:
            bbu = [c for c in bb.columns if 'BBU' in c]
            bbl = [c for c in bb.columns if 'BBL' in c]
            if bbu and bbl:
                bw         = (bb[bbu[0]] - bb[bbl[0]]) / df['close']
                bb_squeeze = bw.iloc[-1] < bw.iloc[-6:-1].mean()

        # VWAP Proxy
        tp   = (df['high'] + df['low'] + df['close']) / 3
        vwap = (tp * df['volume']).rolling(5).sum() / df['volume'].rolling(5).sum()
        price_above_vwap = df['close'].iloc[-1] > vwap.iloc[-1]

        # 10-Day Tight Range
        last10          = df.tail(10)
        h10, l10        = last10['high'].max(), last10['low'].min()
        tight_range_pct = ((h10 - l10) / l10) * 100 if l10 > 0 else 999

        # Close > 10-day High (prior 9 sessions)
        prior9_high        = df['high'].iloc[-11:-1].max() if len(df) >= 11 else h10
        close_above_10d_h  = df['close'].iloc[-1] > prior9_high

        # Volume Spike
        vol_ratio = df['volume'].iloc[-1] / df['volume'].iloc[-21:-1].mean()

        # Bullish Candle Body
        t          = df.iloc[-1]
        rng        = t['high'] - t['low']
        body       = t['close'] - t['open']
        body_pct   = (body / rng * 100) if rng > 0 else 0
        is_bullish = (body > 0) and (body_pct >= CONFIG['candle_body_pct'])

        return {
            'close': round(df['close'].iloc[-1], 2),
            'atr_today': round(atr_today, 2),
            'atr_contracting': bool(atr_contracting),
            'adx': round(adx_today, 2),
            'adx_turning_up': bool(adx_turning_up),
            'rsi': round(rsi_today, 2),
            'tight_range_pct': round(tight_range_pct, 2),
            'close_above_10d_high': bool(close_above_10d_h),
            'vol_ratio': round(vol_ratio, 2),
            'price_above_ema20': bool(price_above_ema20),
            'bb_squeeze': bool(bb_squeeze),
            'price_above_vwap': bool(price_above_vwap),
            'body_pct': round(body_pct, 2),
            'is_bullish_candle': bool(is_bullish),
            '10d_high': round(h10, 2),
            '10d_low': round(l10, 2),
        }
    except Exception:
        return None

print('Indicator engine ready.')

Indicator engine ready.


In [9]:
def compute_btst_score(ind, deliv_pct):
    score = 0.0
    score += CONFIG['w_vol_spike']        * min(ind['vol_ratio'] / 6.0, 1.0)
    rsi = ind['rsi']
    rsi_score = (1.0 - abs(rsi - 62.5) / 7.5) if CONFIG['rsi_min'] <= rsi <= CONFIG['rsi_max'] else 0
    score += CONFIG['w_rsi']              * rsi_score
    score += CONFIG['w_tight_range']      * max(0, 1 - ind['tight_range_pct'] / CONFIG['tight_range_pct'])
    if ind['adx'] <= CONFIG['adx_max'] and ind['adx_turning_up']:
        score += CONFIG['w_adx']          * (1 - ind['adx'] / CONFIG['adx_max'])
    score += CONFIG['w_atr_contract']     * (1 if ind['atr_contracting']      else 0)
    score += CONFIG['w_price_vs_10dhigh'] * (1 if ind['close_above_10d_high'] else 0)
    score += CONFIG['w_ema_align']        * (1 if ind['price_above_ema20']    else 0)
    score += CONFIG['w_candle_body']      * (1 if ind['is_bullish_candle']    else 0)
    score += CONFIG['w_bb_squeeze']       * (1 if ind['bb_squeeze']           else 0)
    if not np.isnan(deliv_pct) and deliv_pct >= CONFIG['delivery_pct_min']:
        score += CONFIG['w_delivery']     * min(deliv_pct / 80.0, 1.0)
    return round(score, 2)

print('Scoring engine ready.')

Scoring engine ready.


In [10]:
results = []

print(f"Scoring {len(valid_symbols)} symbols...")
deliv_lookup   = dict(zip(deliv_df['SYMBOL'], deliv_df['DELIV_PCT'])) if not deliv_df.empty else {}
results        = []
failed_reasons = {}
for sym in tqdm(valid_symbols, desc="Scoring"):

    df_sym = ohlcv_cache.get(sym)
    if df_sym is None or len(df_sym) < 25:
        continue  # structural filter only

    ind = compute_indicators(df_sym)
    if ind is None:
        continue  # bad data filter only

    deliv_pct = deliv_lookup.get(sym, np.nan)

    # ---- Score Everything ----
    score = compute_btst_score(ind, deliv_pct)

    # Optional minimum threshold (recommended)
    if score < CONFIG.get("min_score", 0):
        continue

    results.append({
        'Symbol'      : sym,
        'Score'       : score,
        'Close'       : ind['close'],
        'RSI'         : ind['rsi'],
        'ADX'         : ind['adx'],
        'VolRatio'    : ind['vol_ratio'],
        'ATR'         : ind['atr_today'],
        'TightRange%' : ind['tight_range_pct'],
        'ATR_Cont'    : 'Y' if ind['atr_contracting'] else 'N',
        'ADX_Up'      : 'Y' if ind['adx_turning_up'] else 'N',
        'Close_10dH'  : 'Y' if ind['close_above_10d_high'] else 'N',
        'EMA20'       : 'Y' if ind['price_above_ema20'] else 'N',
        'BB_Sqz'      : 'Y' if ind['bb_squeeze'] else 'N',
        'BullCdl'     : 'Y' if ind['is_bullish_candle'] else 'N',
        'Deliv%'      : round(deliv_pct, 1) if not np.isnan(deliv_pct) else np.nan,
        'Mcap_Cr'     : mcap_cache.get(sym),
    })

# ---- Convert to DataFrame ----
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠ No stocks scored above threshold.")
else:
    results_df = (
        results_df
        .sort_values('Score', ascending=False)
        .reset_index(drop=True)
    )

    top_df = results_df.head(CONFIG['top_n']).copy()
    top_df.insert(0, 'Rank', range(1, len(top_df) + 1))

    print(f"Total scored: {len(results_df)} | Showing Top {len(top_df)}")

Scoring 824 symbols...


Scoring:   0%|          | 0/824 [00:00<?, ?it/s]

Total scored: 822 | Showing Top 15


In [11]:
'''def passes_hard_filters(ind):
    fails = []
    if ind['tight_range_pct']   >= CONFIG['tight_range_pct']:      fails.append(f"TightRange {ind['tight_range_pct']:.1f}% >= 6%")
    if not ind['atr_contracting']:                                   fails.append('ATR not contracting')
    if ind['adx']               >= CONFIG['adx_max']:               fails.append(f"ADX {ind['adx']:.1f} >= 15")
    if not ind['adx_turning_up']:                                    fails.append('ADX not turning up')
    if ind['vol_ratio']         < CONFIG['vol_spike_mult']:         fails.append(f"VolRatio {ind['vol_ratio']:.1f}x < 3x")
    if not ind['close_above_10d_high']:                              fails.append('Not above 10d high')
    if not (CONFIG['rsi_min'] <= ind['rsi'] <= CONFIG['rsi_max']):  fails.append(f"RSI {ind['rsi']:.1f} outside 55-70")
    return len(fails) == 0, fails

deliv_lookup   = dict(zip(deliv_df['SYMBOL'], deliv_df['DELIV_PCT'])) if not deliv_df.empty else {}
results        = []
failed_reasons = {}

print(f'Screening {len(valid_symbols)} symbols...')

for sym in tqdm(valid_symbols, desc='Screening'):
    df_sym = ohlcv_cache.get(sym)
    if df_sym is None or len(df_sym) < 25:
        continue
    ind = compute_indicators(df_sym)
    if ind is None:
        continue
    passed, fails = passes_hard_filters(ind)
    if not passed:
        failed_reasons[sym] = fails
        continue

    deliv_pct  = deliv_lookup.get(sym, np.nan)
    results.append({
        'Symbol'       : sym,
        'Score'        : compute_btst_score(ind, deliv_pct),
        'Close'        : ind['close'],
        'RSI'          : ind['rsi'],
        'ADX'          : ind['adx'],
        
        'ADX_Up'       : 'Y' if ind['adx_turning_up']      else 'N',
        'ATR_Cont'     : 'Y' if ind['atr_contracting']     else 'N',
        'TightRange%'  : ind['tight_range_pct'],
        'VolRatio'     : ind['vol_ratio'],
        'Close_10dH'   : 'Y' if ind['close_above_10d_high'] else 'N',
        'EMA20'        : 'Y' if ind['price_above_ema20']   else 'N',
        'BB_Sqz'       : 'Y' if ind['bb_squeeze']          else 'N',
        'VWAP'         : 'Y' if ind['price_above_vwap']    else 'N',
        'BullCdl'      : 'Y' if ind['is_bullish_candle']   else 'N',
        'Body%'        : ind['body_pct'],
        'Deliv%'       : round(deliv_pct, 1) if not np.isnan(deliv_pct) else np.nan,
        'Mcap_Cr'      : mcap_cache.get(sym),
        '10d_High'     : ind['10d_high'],
        'ATR'          : ind['atr_today'],
    })

print(results)
results_df = pd.DataFrame(results).sort_values('Score', ascending=False).reset_index(drop=True)
top_df     = results_df.head(CONFIG['top_n']).copy()
top_df.insert(0, 'Rank', range(1, len(top_df)+1))

print(f'Passed all filters: {len(results_df)} | Top {len(top_df)} shown')'''

'def passes_hard_filters(ind):\n    fails = []\n    if ind[\'tight_range_pct\']   >= CONFIG[\'tight_range_pct\']:      fails.append(f"TightRange {ind[\'tight_range_pct\']:.1f}% >= 6%")\n    if not ind[\'atr_contracting\']:                                   fails.append(\'ATR not contracting\')\n    if ind[\'adx\']               >= CONFIG[\'adx_max\']:               fails.append(f"ADX {ind[\'adx\']:.1f} >= 15")\n    if not ind[\'adx_turning_up\']:                                    fails.append(\'ADX not turning up\')\n    if ind[\'vol_ratio\']         < CONFIG[\'vol_spike_mult\']:         fails.append(f"VolRatio {ind[\'vol_ratio\']:.1f}x < 3x")\n    if not ind[\'close_above_10d_high\']:                              fails.append(\'Not above 10d high\')\n    if not (CONFIG[\'rsi_min\'] <= ind[\'rsi\'] <= CONFIG[\'rsi_max\']):  fails.append(f"RSI {ind[\'rsi\']:.1f} outside 55-70")\n    return len(fails) == 0, fails\n\ndeliv_lookup   = dict(zip(deliv_df[\'SYMBOL\'], deliv_df[\'DELIV_PCT\']

In [12]:
# Styled table
display(top_df.style
    .map(lambda v: 'background:#1a472a;color:white' if v>=80 else ('background:#2d6a4f;color:white' if v>=65 else ''), subset=['Score'])
    .format({'Score':'{:.1f}','Close':'{:.2f}','RSI':'{:.1f}','ADX':'{:.1f}','TightRange%':'{:.2f}','VolRatio':'{:.1f}'}, na_rep='N/A')
)

# Score bar chart
fig = px.bar(top_df.sort_values('Score'), x='Score', y='Symbol', orientation='h',
             color='Score', color_continuous_scale='Viridis', text='Score',
             title=f'BTST Scores — {SCAN_DATE}',
             hover_data=['Close','RSI','ADX','VolRatio'])
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(plot_bgcolor='#0d1117', paper_bgcolor='#0d1117', font_color='white',
                  height=max(400, len(top_df)*35), coloraxis_showscale=False)
fig.show()

,Rank,Symbol,Score,Close,RSI,ADX,VolRatio,ATR,TightRange%,ATR_Cont,ADX_Up,Close_10dH,EMA20,BB_Sqz,BullCdl,Deliv%,Mcap_Cr
0,1,HGS,49.8,397.95,54.0,38.9,19.3,16.080000,19.28,N,N,Y,Y,Y,Y,76.300000,1842.850000
1,2,JAGSNPHARM,49.2,180.04,57.5,16.5,25.5,8.150000,22.18,N,Y,Y,Y,Y,N,65.500000,1207.120000
2,3,WONDERLA,46.1,520.25,63.1,16.9,1.1,15.700000,15.46,N,N,Y,Y,Y,Y,56.000000,3306.220000
3,4,KSHINTL,40.8,388.50,58.0,12.1,3.5,16.750000,13.10,N,N,Y,Y,N,Y,49.600000,2632.310000
4,5,RSYSTEMS,40.0,327.95,54.7,37.6,12.2,20.210000,33.45,N,N,Y,Y,Y,N,5.900000,3894.540000
5,6,STYL,35.8,267.50,60.3,21.8,1.6,12.280000,21.02,N,N,Y,Y,Y,N,22.700000,4331.380000
6,7,BOROLTD,35.1,254.11,55.7,23.8,21.8,12.200000,22.14,N,Y,N,Y,N,Y,58.500000,3032.240000
7,8,STALLION,35.0,106.18,29.7,34.0,9.4,9.030000,35.12,Y,Y,N,N,N,N,100.000000,1237.470000
8,9,GULPOLY,33.4,158.58,60.0,32.7,0.1,6.660000,13.71,Y,Y,N,Y,Y,N,49.000000,987.730000
9,10,SMSPHARMA,32.7,393.90,66.5,45.0,0.2,17.370000,11.77,Y,N,N,Y,Y,Y,35.200000,3680.520000


In [13]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [16]:
for sym in top_df.head(5)['Symbol'].tolist():
    df_p = ohlcv_cache.get(sym, pd.DataFrame()).tail(30).copy()
    if df_p.empty: continue
    df_p.columns = [c.lower() for c in df_p.columns]
    colors = ['#26a69a' if c>=o else '#ef5350' for c,o in zip(df_p['close'], df_p['open'])]

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75,0.25], vertical_spacing=0.03)
    fig.add_trace(go.Candlestick(x=df_p.index, open=df_p['open'], high=df_p['high'],
        low=df_p['low'], close=df_p['close'],
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350', name=sym), row=1, col=1)

    ema20 = ta.ema(df_p['close'], length=20)
    if ema20 is not None:
        fig.add_trace(go.Scatter(x=df_p.index, y=ema20, line=dict(color='orange',width=1.5,dash='dot'), name='EMA20'), row=1, col=1)

    fig.add_trace(go.Bar(x=df_p.index, y=df_p['volume'], marker_color=colors, showlegend=False), row=2, col=1)
    avg_v = df_p['volume'].rolling(20).mean()
    fig.add_trace(go.Scatter(x=df_p.index, y=avg_v, line=dict(color='yellow',width=1,dash='dash'), showlegend=False), row=2, col=1)

    score_val = top_df[top_df['Symbol']==sym]['Score'].values[0]
    fig.update_layout(title=f'{sym} | Score: {score_val:.0f}/100', plot_bgcolor='#0d1117',
                      paper_bgcolor='#0d1117', font_color='white', height=500, xaxis_rangeslider_visible=False)
    print("Plotting:", sym)
    fig.show()

Plotting: HGS


Plotting: JAGSNPHARM


Plotting: WONDERLA


Plotting: KSHINTL


Plotting: RSYSTEMS


In [15]:
print(f'BTST TRADE PLANS — {SCAN_DATE}')
print(f'Portfolio: Rs{PORTFOLIO_VALUE:,.0f} | Risk/Trade: {MAX_RISK_PER_TRADE_PCT}%\n')

def compute_dynamic_target(ind, entry_price):

    atr = ind["atr_today"]
    adx = ind["adx"]
    vol_ratio = ind["vol_ratio"]

    # ---- Base Target: 1.5x ATR ----
    target_multiplier = 1

    # ---- Strong Trend Bonus ----
    if adx > 20:
        target_multiplier += 0.5
    if adx > 25:
        target_multiplier += 0.5

    # ---- Volume Expansion Bonus ----
    if vol_ratio > 1.5:
        target_multiplier += 0.5
    if vol_ratio > 2.0:
        target_multiplier += 0.5

    # ---- Cap Extreme Targets ----
    target_multiplier = min(target_multiplier, 3)

    target_price = entry_price + (target_multiplier * atr)

    expected_return = (target_price - entry_price) / entry_price

    return target_price, expected_return

trade_plans = []
for _, row in top_df.iterrows():
    sym, close, atr_v = row['Symbol'], row['Close'], row['ATR']
    entry     = round(close * 1.002, 2)
    target, upside = compute_dynamic_target(ind, entry)
    #target    = round(entry * 1.055, 2)
    stop      = round(entry - 1.5 * atr_v, 2)
    risk_pt   = max(entry - stop, 0.01)
    rr        = round((target - entry) / risk_pt, 2)
    qty       = max(int(PORTFOLIO_VALUE * MAX_RISK_PER_TRADE_PCT / 100 / risk_pt), 1)
    capital   = round(qty * entry, 2)

    trade_plans.append({'Symbol':sym, 'Score':row['Score'],
        'Entry~':entry, 'Target':target, 'StopLoss':stop, 'R:R':rr,
        'Qty':qty, 'Capital_Rs':capital, 'Risk_Rs':round(qty*risk_pt,2),
        'Upside%': round(upside*100,2) ,
        'Downside%':round((entry-stop)/entry*100,2)})

    print(f"{sym} (Score:{row['Score']:.0f}) | Entry:Rs{entry} | Target:Rs{target} (+5.5%) | Stop:Rs{stop} | R:R={rr}:1 | Qty:{qty} | Capital:Rs{capital:,.0f}")

trade_df = pd.DataFrame(trade_plans)
display(trade_df)

BTST TRADE PLANS — 2026-03-09
Portfolio: Rs500,000 | Risk/Trade: 2.0%

HGS (Score:50) | Entry:Rs398.75 | Target:Rs423.25 (+5.5%) | Stop:Rs374.63 | R:R=1.02:1 | Qty:414 | Capital:Rs165,082
JAGSNPHARM (Score:49) | Entry:Rs180.4 | Target:Rs204.9 (+5.5%) | Stop:Rs168.18 | R:R=2.0:1 | Qty:818 | Capital:Rs147,567
WONDERLA (Score:46) | Entry:Rs521.29 | Target:Rs545.79 (+5.5%) | Stop:Rs497.74 | R:R=1.04:1 | Qty:424 | Capital:Rs221,027
KSHINTL (Score:41) | Entry:Rs389.28 | Target:Rs413.78 (+5.5%) | Stop:Rs364.15 | R:R=0.97:1 | Qty:397 | Capital:Rs154,544
RSYSTEMS (Score:40) | Entry:Rs328.61 | Target:Rs353.11 (+5.5%) | Stop:Rs298.3 | R:R=0.81:1 | Qty:329 | Capital:Rs108,113
STYL (Score:36) | Entry:Rs268.04 | Target:Rs292.54 (+5.5%) | Stop:Rs249.62 | R:R=1.33:1 | Qty:542 | Capital:Rs145,278
BOROLTD (Score:35) | Entry:Rs254.62 | Target:Rs279.12 (+5.5%) | Stop:Rs236.32 | R:R=1.34:1 | Qty:546 | Capital:Rs139,023
STALLION (Score:35) | Entry:Rs106.39 | Target:Rs130.89 (+5.5%) | Stop:Rs92.84 | R:R=1.81

,Symbol,Score,Entry~,Target,StopLoss,R:R,Qty,Capital_Rs,Risk_Rs,Upside%,Downside%
0,HGS,49.77,398.75,423.25,374.63,1.02,414,165082.50,9985.68,6.14,6.05
1,JAGSNPHARM,49.16,180.40,204.90,168.18,2.00,818,147567.20,9995.96,13.58,6.77
2,WONDERLA,46.08,521.29,545.79,497.74,1.04,424,221026.96,9985.20,4.70,4.52
3,KSHINTL,40.82,389.28,413.78,364.15,0.97,397,154544.16,9976.61,6.29,6.46
4,RSYSTEMS,40.00,328.61,353.11,298.30,0.81,329,108112.69,9971.99,7.46,9.22
5,STYL,35.81,268.04,292.54,249.62,1.33,542,145277.68,9983.64,9.14,6.87
6,BOROLTD,35.10,254.62,279.12,236.32,1.34,546,139022.52,9991.80,9.62,7.19
7,STALLION,35.00,106.39,130.89,92.84,1.81,738,78515.82,9999.90,23.03,12.74
8,GULPOLY,33.36,158.90,183.40,148.91,2.45,1001,159058.90,9999.99,15.42,6.29
9,SMSPHARMA,32.74,394.69,419.19,368.63,0.94,383,151166.27,9980.98,6.21,6.60
